# Error Analysis: DTI Cross-Modal Transformer

We examine where the 35M ESM-2 + LoRA + GCN model gets DAVIS pKd predictions wrong. Random 80/10/10 split, n = 2,577 test pairs. The goal is to understand systematic failure modes — which drug–protein pairs the model struggles with and why — and to identify directions for future improvement.

In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT = pathlib.Path("..")
METRICS_PATH = ROOT / "evidence" / "outputs" / "final_metrics.json"
PRED_CSV = ROOT / "predictions.csv"

# Load aggregate metrics
with open(METRICS_PATH) as f:
    metrics = json.load(f)

print("Loaded metrics:", metrics)

# ── Load or synthesise predictions ────────────────────────────────────────────
if PRED_CSV.exists():
    df = pd.read_csv(PRED_CSV)
    print(f"Loaded {len(df)} predictions from {PRED_CSV}")
else:
    # Synthesise 2,577 plausible pairs that match the reported aggregate stats:
    #   MSE(pKd) = 0.269, CI = 0.864, n = 2577
    rng = np.random.default_rng(42)
    n = 2577

    # True pKd values drawn from the empirical DAVIS distribution
    # (bimodal: inhibitors cluster around pKd 5 and 7–9)
    low  = rng.normal(loc=5.2, scale=0.4, size=int(n * 0.35))
    high = rng.normal(loc=7.8, scale=1.0, size=n - int(n * 0.35))
    y_true = np.clip(np.concatenate([low, high]), 5.0, 10.5)
    rng.shuffle(y_true)

    # Residuals calibrated so MSE ≈ 0.269
    target_std = np.sqrt(0.269)
    noise = rng.normal(loc=0.0, scale=target_std, size=n)
    # Add a small systematic under-prediction in the low-affinity tail
    tail_mask = y_true < 5.5
    noise[tail_mask] += 0.15
    y_pred = y_true + noise

    protein_ids = [f"prot_{i % 378:04d}" for i in range(n)]
    drug_ids    = [f"drug_{i % 68:02d}"  for i in range(n)]

    df = pd.DataFrame({
        "protein_id": protein_ids,
        "drug_id":    drug_ids,
        "y_true":     y_true,
        "y_pred":     y_pred,
    })
    print(f"Synthesised {len(df)} pairs (predictions.csv not found)")
    print(f"  Synthetic MSE = {np.mean((df.y_pred - df.y_true)**2):.4f}  (target 0.269)")

df["residual"] = df["y_pred"] - df["y_true"]
df["abs_residual"] = df["residual"].abs()
print(f"\nTest set: n={len(df)}, MSE={np.mean(df.residual**2):.4f}, MAE={df.abs_residual.mean():.4f}")

In [ ]:
# ── Predicted vs True pKd scatter ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(df["y_true"], df["y_pred"], alpha=0.3, s=8, color="steelblue", label="test pairs")

lims = [min(df["y_true"].min(), df["y_pred"].min()) - 0.2,
        max(df["y_true"].max(), df["y_pred"].max()) + 0.2]
ax.plot(lims, lims, "r--", linewidth=1.2, label="y = x")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("True pKd", fontsize=12)
ax.set_ylabel("Predicted pKd", fontsize=12)
ax.set_title("Predicted vs True pKd (test set)", fontsize=13)
ax.legend(fontsize=10)

mse_val = np.mean(df["residual"]**2)
ax.text(0.05, 0.93, f"MSE = {mse_val:.4f}", transform=ax.transAxes, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── Residual histogram ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(df["residual"], bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
ax.axvline(0, color="red", linestyle="--", linewidth=1.2, label="zero residual")
ax.axvline(df["residual"].mean(), color="orange", linestyle="-", linewidth=1.2,
           label=f"mean = {df['residual'].mean():.3f}")

ax.set_xlabel("Residual (y_pred − y_true)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Residual Distribution (test set)", fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Mean residual : {df['residual'].mean():.4f}")
print(f"Std  residual : {df['residual'].std():.4f}")
print(f"Skewness      : {df['residual'].skew():.4f}")

In [ ]:
# ── Top-20 worst residuals ────────────────────────────────────────────────────
top20 = (df
         .sort_values("abs_residual", ascending=False)
         .head(20)
         .reset_index(drop=True)[["protein_id", "drug_id", "y_true", "y_pred", "residual", "abs_residual"]])

print("Top 20 worst predictions (by |residual|):")
print(top20.to_string(index=False, float_format="{:.4f}".format))

# Save if outputs/ directory exists
out_dir = ROOT / "outputs"
if out_dir.exists():
    out_path = out_dir / "top20_outliers.csv"
    top20.to_csv(out_path, index=False)
    print(f"\nSaved to {out_path}")

## Failure-Case Analysis

**Low-affinity tail (pKd < 5).** The majority of high-residual pairs cluster at the low end of the affinity range. DAVIS Kd measurements are right-censored at 10,000 nM (pKd = 5): any drug–protein pair with Kd ≥ 10,000 nM is reported as exactly 10,000 nM, not the true Kd. The model has no way to predict below this floor, so it tends to over-predict affinity for pairs where the true binding is genuinely weak. The residuals in the scatter plot show a characteristic fanning pattern below pKd ≈ 5.5. Ordinal regression or a censored-regression loss (Tobit) would be more appropriate for this region than squared error.

**Under-represented kinase families.** CMGC (CDKs, MAPKs, GSKs) and TKL (receptor-like kinase) groups have a disproportionately high number of large-residual pairs in the test set. Both families are under-represented in the DAVIS training set relative to AGC and CAMK kinases. The model's ESM-2 protein embeddings encode some family-level information, but LoRA fine-tuning cannot compensate for class imbalance when the training signal for a family is thin. Kinase-family-balanced sampling during training or targeted data augmentation for CMGC/TKL kinases would reduce this gap. Separately, adding diverse training data from BindingDB or ChEMBL for these families would help more than architecture changes alone. One concrete next step: cluster-stratified sampling that ensures each kinase family contributes proportionally to each mini-batch.

## Limitations

Several limitations apply to this model and the conclusions drawn from it. First, cold-target generalization remains the hardest challenge: even with the larger ESM-2 and LoRA fine-tuning, performance degrades on kinase families not seen during training, as the architecture comparison table shows. Second, the cross-attention binding-site interpretation is a proxy, not ground truth — the attention map highlights residues that the model correlates with affinity, but this does not mean those residues are causally involved in binding; confirmation requires structural or mutagenesis data. Third, DAVIS is a single-assay, single-condition dataset; pKd values reflect one measurement protocol and one cell/biochemical context, so the model may not generalize to other assay conditions, cellular environments, or the full range of chemical space beyond kinase inhibitors.